# Bước 2: Đánh giá lại SP-Det trên tập test chính thức của VinDr-CXR

Notebook này làm các mục **2.1, 2.2, 2.3** trong task list (Bảng 7 của review, bước 2):

- **2.1** Train trên đủ 15,000 ảnh train (gồm cả ảnh bình thường), test trên 3,000 ảnh test chính thức có nhãn đồng thuận của 5 bác sĩ. Hộp của 3 bác sĩ ở tập train được gộp bằng WBF.
- **2.2** Bảng SP-Det và YOLOv8s theo quy trình mới, đủ chỉ số Bảng 3 của review, chạy nhiều seed, có khoảng tin cậy và kiểm định cặp.
- **2.3** Bảng đặt cạnh nhau: kết quả trên cách chia 8:1:1 (bước 1) và trên tập test chính thức.

So với bước 1: tập val là 10% của 15,000 ảnh, giữ tỉ lệ ảnh bình thường. Mọi lựa chọn khác giữ nguyên (`configs/common.yaml`, tắt mosaic và lật ảnh, imgsz 640, WBF IoU 0.4).

**Cần có:**
- Dữ liệu Kaggle VinBigData: `train.csv`, `train/` (15,000 DICOM), `test/` (3,000 DICOM).
- File nhãn test của PhysioNet (`annotations_test.csv`) — cần tài khoản PhysioNet đã được duyệt.

In [ ]:
# Cài một lần (bỏ dấu # nếu máy chưa có): giống notebook 01
# %pip install ultralytics==8.4.171 ensemble-boxes pydicom pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg spacy pandas pyyaml
# %pip install "transformers==4.40.0" accelerate bitsandbytes
# !python -m spacy download en_core_web_sm

## 0. Cấu hình

In [ ]:
import os, sys, json, subprocess
from pathlib import Path
import numpy as np, pandas as pd, yaml, torch

SPDET_ROOT = Path("..").resolve()
KAGGLE_DIR = Path("/data/vinbigdata")                 # train.csv, train/, test/
TEST_LABELS = Path("/data/physionet/annotations_test.csv")   # nhãn test chính thức (PhysioNet)
WORK = Path("/data/spdet_work")                       # dùng chung với notebook 01 (dùng lại PNG, báo cáo đã sinh)
DEVICE = 0
SEEDS = [0, 1, 2, 3, 4]                               # review: nhiều seed; giảm nếu thiếu GPU
VAL_SEED = 42
RUN = {"yolov8s": True, "spdet": True, "yolo_world": False}   # 2.2 chỉ yêu cầu SP-Det và YOLOv8s
LOW_VRAM = False

sys.path.insert(0, str(SPDET_ROOT)); sys.path.insert(0, str(SPDET_ROOT / "eval"))
import prep, cxr_eval
CFG = yaml.safe_load(open(SPDET_ROOT / "configs" / "common.yaml"))
TRAIN_ARGS = dict(CFG["train"]); TRAIN_ARGS.update(mosaic=0.0, fliplr=0.0)
if LOW_VRAM:
    TRAIN_ARGS.update(batch=8, workers=4)   # Ultralytics vẫn cộng dồn gradient tới 64 ảnh
PNG_DIR, PNG_TEST = WORK / "png", WORK / "png_test"
DS_DIR, PROMPT_DIR, RES_DIR = WORK / "yolo_official", WORK / "prompts", WORK / "results_step2"
for d in (PNG_DIR, PNG_TEST, DS_DIR, PROMPT_DIR, RES_DIR):
    d.mkdir(parents=True, exist_ok=True)
import ultralytics
assert ultralytics.__version__ == CFG["ultralytics_version"], "Sai phiên bản ultralytics so với configs/common.yaml"

## 1. Dữ liệu (mục 2.1)

In [ ]:
# Train: 15,000 ảnh, gộp hộp 3 bác sĩ bằng WBF; ảnh bình thường có file nhãn rỗng
rows_tr = prep.read_kaggle_train(KAGGLE_DIR / "train.csv")
train_ids = sorted(rows_tr)
pos_ids = {i for i, b in rows_tr.items() if b}
print(f"Train: {len(train_ids)} ảnh, {len(pos_ids)} có tổn thương, {len(train_ids) - len(pos_ids)} bình thường")
meta = prep.convert_all(train_ids, KAGGLE_DIR / "train", PNG_DIR, WORK / "meta.csv", workers=8)

# Test: nhãn chính thức
rows_te, unmatched = prep.read_vindr_annotations(TEST_LABELS)
test_ids = sorted(rows_te)
print(f"Test: {len(test_ids)} ảnh, {sum(1 for b in rows_te.values() if b)} có tổn thương")
if unmatched:
    print("Nhãn không thuộc 14 lớp, đã bỏ (kiểm tra lại tên lớp):", unmatched)
meta_te = prep.convert_all(test_ids, KAGGLE_DIR / "test", PNG_TEST, WORK / "meta_test.csv", workers=8)

In [ ]:
fused_tr = prep.fuse_radiologists(rows_tr, meta, iou_thr=CFG["data"]["wbf_iou_thr"])
n_rads = {len({r[0] for r in b}) for b in rows_te.values() if b}
# Nhãn test của PhysioNet là nhãn đồng thuận (một bộ hộp mỗi ảnh); nếu có nhiều rad_id thì cũng gộp bằng WBF
fused_te = prep.fuse_radiologists(rows_te, meta_te, iou_thr=CFG["data"]["wbf_iou_thr"]) if max(n_rads, default=1) > 1 \
    else {i: [(c, x1, y1, x2, y2) for _, c, x1, y1, x2, y2 in b] for i, b in rows_te.items()}
print("Số người gán nhãn mỗi ảnh test:", n_rads)

split_file = WORK / "split_official.json"
if split_file.exists():
    splits = json.load(open(split_file))
else:
    splits = prep.split_train_val(train_ids, pos_ids, val_frac=0.1, seed=VAL_SEED)
    splits["test"] = test_ids
    json.dump(splits, open(split_file, "w"))
fused = {**fused_tr, **fused_te}
meta_all = {**meta, **meta_te}

# Ảnh test nằm ở PNG_TEST: tạo dataset YOLO cho train/val và test riêng rồi gộp data.yaml
DATA_YAML = prep.write_yolo_dataset(DS_DIR, {"train": splits["train"], "val": splits["val"]}, fused, meta_all, PNG_DIR)
prep.write_yolo_dataset(DS_DIR, {"test": splits["test"]}, fused, meta_all, PNG_TEST)
d = yaml.safe_load(open(DS_DIR / "data.yaml")); d.update(train="images/train", val="images/val", test="images/test")
yaml.safe_dump(d, open(DATA_YAML, "w"), sort_keys=False)
prep.write_gt_csv(fused, splits["test"], RES_DIR / "gt_test.csv")

counts = pd.DataFrame({k: prep.class_counts(fused, v) for k, v in splits.items()}, index=prep.VINDR_CLASSES)
counts.loc["Số ảnh"] = [len(v) for v in splits.values()]
counts.loc["Ảnh bình thường"] = [sum(1 for i in v if not fused[i]) for v in splits.values()]
counts

## 2. Gợi ý: báo cáo CheXagent, tên bệnh, mã hoá (dùng lại phần đã sinh ở notebook 01)

In [ ]:
REPORTS = WORK / "reports.jsonl"
r_tr = prep.run_reports(train_ids, PNG_DIR, REPORTS, quant4=LOW_VRAM)
reports = prep.run_reports(test_ids, PNG_TEST, REPORTS, quant4=LOW_VRAM)
torch.cuda.empty_cache()
print(len(reports), "báo cáo")

In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm")
DBP_FILE = WORK / "dbp.json"
dbp = json.load(open(DBP_FILE)) if DBP_FILE.exists() else {}
for iid, rep in reports.items():
    if iid not in dbp:
        dbp[iid] = prep.extract_dbp(rep, nlp)
json.dump(dbp, open(DBP_FILE, "w"))

# Chất lượng gợi ý trên test: lớp được gợi ý có đúng là lớp ảnh có không; và ảnh bình thường bị gợi ý bệnh bao nhiêu
name2id = {n: i for i, n in enumerate(prep.VINDR_CLASSES)}
tp = fp = fn = 0; normal_flagged = 0; n_normal = 0
for iid in splits["test"]:
    pred = {name2id[c] for c in dbp[iid]["positives"]}; true = {c for c, *_ in fused[iid]}
    tp += len(pred & true); fp += len(pred - true); fn += len(true - pred)
    if not true:
        n_normal += 1; normal_flagged += bool(pred)
print(f"Test: precision {tp / max(tp + fp, 1):.2f}, recall {tp / max(tp + fn, 1):.2f}; "
      f"{normal_flagged}/{n_normal} ảnh bình thường được gợi ý ít nhất một bệnh")

In [ ]:
todo = {k: v for k, v in reports.items() if not (PROMPT_DIR / f"{k}.pt").exists()}
prep.encode_prompts(todo, {k: dbp[k]["prompts"] for k in todo}, PROMPT_DIR,
                    device="cuda" if torch.cuda.is_available() else "cpu")
print(len(list(PROMPT_DIR.glob("*.pt"))), "file gợi ý")

## 3. Train (mục 2.2): mỗi mô hình × mỗi seed

In [ ]:
from ultralytics import YOLO, YOLOWorld
import spdet_model as S
S.CTX.store = S.PromptStore(PROMPT_DIR)
RUNS = WORK / "runs_step2"

def train(kind, seed):
    name = f"{kind}_s{seed}"
    best = RUNS / name / "weights" / "best.pt"
    if best.exists():
        print("Đã có", best); return best
    args = dict(data=str(DATA_YAML), seed=seed, device=DEVICE, project=str(RUNS), name=name, exist_ok=True, **TRAIN_ARGS)
    if kind == "yolov8s":
        m = YOLO("yolov8s.pt"); m.train(**args)
    elif kind == "yolo_world":
        m = YOLOWorld("yolov8s-worldv2.pt"); m.train(**args)
    else:
        m = YOLOWorld("yolov8s-worldv2.pt"); m.train(trainer=S.SPDetTrainer, **args)
    return Path(m.trainer.best)

BEST = {(k, s): train(k, s) for s in SEEDS for k, on in RUN.items() if on}
BEST

## 4. Đánh giá trên 3,000 ảnh test (mục 2.2)

Mỗi lượt chạy xuất dự đoán (conf 0.001, tối đa 100 hộp mỗi ảnh) theo toạ độ ảnh gốc, rồi chấm bằng `eval/cxr_eval.py` với đủ chỉ số Bảng 3: mAP40, mAP40:95, mAP50, mAP50:95, AP theo kích thước, AP từng lớp kèm số hộp, FROC, AUC mức ảnh, D-ECE, khoảng tin cậy bootstrap 1,000 lần.

In [ ]:
test_imgs = [DS_DIR / "images" / "test" / f"{i}.png" for i in splits["test"]]
ids_file = RES_DIR / "gt_test_ids.txt"
EVAL = {}
for (kind, seed), best in BEST.items():
    out = RES_DIR / f"{kind}_s{seed}"
    pred_csv = RES_DIR / f"pred_{kind}_s{seed}.csv"
    if not pred_csv.exists():
        model = YOLO(str(best)) if kind == "yolov8s" else YOLOWorld(str(best))
        S.export_predictions(model, test_imgs, meta_all, pred_csv, use_prompts=kind == "spdet",
                             imgsz=TRAIN_ARGS["imgsz"], device=DEVICE)
    if not Path(str(out) + ".json").exists():
        subprocess.run([sys.executable, str(SPDET_ROOT / "eval" / "cxr_eval.py"), "--gt", str(RES_DIR / "gt_test.csv"),
                        "--pred", str(pred_csv), "--images", str(ids_file), "--out", str(out)], check=True,
                       stdout=subprocess.DEVNULL)
    EVAL[(kind, seed)] = json.load(open(str(out) + ".json"))["configs"]["A"]

# Bảng chính: trung bình ± độ lệch chuẩn qua các seed
rows = {}
for kind in [k for k, on in RUN.items() if on]:
    ms = [EVAL[(kind, s)]["metrics"] for s in SEEDS if (kind, s) in EVAL]
    sizes = [EVAL[(kind, s)]["ap_by_size@0.4"] for s in SEEDS if (kind, s) in EVAL]
    row = {}
    for m in ms[0]:
        v = np.array([x[m] for x in ms], float) * (1 if m == "D-ECE@0.4" else 100)
        row[m] = f"{np.nanmean(v):.{3 if m == 'D-ECE@0.4' else 1}f} ± {np.nanstd(v):.{3 if m == 'D-ECE@0.4' else 1}f}"
    for sz in sizes[0]:
        v = np.array([x[sz] for x in sizes], float) * 100
        row[f"AP40 {sz}"] = f"{np.nanmean(v):.1f} ± {np.nanstd(v):.1f}"
    rows[f"{kind} ({len(ms)} seed)"] = row
table_main = pd.DataFrame(rows).T
table_main.to_csv(RES_DIR / "table_step2_main.csv")
table_main

In [ ]:
# AP40 từng lớp kèm số hộp test (trung bình qua seed)
first = next(iter(EVAL.values()))
per = {"Số hộp test": [r["n_boxes"] for r in first["per_class@0.4"]]}
for kind in [k for k, on in RUN.items() if on]:
    arr = np.array([[np.nan if r["AP40"] is None else 100 * r["AP40"] for r in EVAL[(kind, s)]["per_class@0.4"]]
                    for s in SEEDS if (kind, s) in EVAL])
    per[kind] = np.round(np.nanmean(arr, 0), 1)
table_cls = pd.DataFrame(per, index=prep.VINDR_CLASSES)
table_cls.to_csv(RES_DIR / "table_step2_per_class.csv")
table_cls

In [ ]:
# Kiểm định cặp SP-Det với YOLOv8s trên cùng ảnh, từng seed (bootstrap 1,000 lần theo ảnh)
paired = []
for s in SEEDS:
    a, b = RES_DIR / f"pred_spdet_s{s}.csv", RES_DIR / f"pred_yolov8s_s{s}.csv"
    if a.exists() and b.exists():
        out = RES_DIR / f"paired_spdet_vs_yolov8s_s{s}"
        if not Path(str(out) + ".json").exists():
            subprocess.run([sys.executable, str(SPDET_ROOT / "eval" / "cxr_eval.py"), "--gt", str(RES_DIR / "gt_test.csv"),
                            "--pred", str(a), "--pred-b", str(b), "--images", str(ids_file), "--out", str(out)],
                           check=True, stdout=subprocess.DEVNULL)
        p = json.load(open(str(out) + ".json"))["paired_A_minus_B"]
        for m in ("mAP40", "mAP40:95", "sens@1FP", "image_AUC"):
            paired.append({"seed": s, "chỉ số": m, "SP-Det trừ YOLOv8s": round(100 * p[m]["diff"], 2),
                           "KTC 95%": f"[{100 * p[m]['ci95'][0]:.2f}, {100 * p[m]['ci95'][1]:.2f}]", "p": round(p[m]["p"], 4)})
table_paired = pd.DataFrame(paired)
table_paired.to_csv(RES_DIR / "table_step2_paired.csv", index=False)
table_paired

## 5. So sánh 8:1:1 với tập test chính thức (mục 2.3)

In [ ]:
step1 = WORK / "results_step1" / "table1_reproduced.csv"
cmp = []
for kind in [k for k, on in RUN.items() if on]:
    ms = [EVAL[(kind, s)]["metrics"] for s in SEEDS if (kind, s) in EVAL]
    r = {"mô hình": kind,
         "AP40 test chính thức": round(100 * np.mean([m["mAP40"] for m in ms]), 1),
         "mAP40:95 test chính thức": round(100 * np.mean([m["mAP40:95"] for m in ms]), 1)}
    if step1.exists():
        t1 = pd.read_csv(step1, index_col=0)
        ours = [i for i in t1.index if i.startswith(kind) and "nhóm" in i]
        paper = f"{kind} (bài)"
        if ours:
            r["AP40 8:1:1 (nhóm)"] = t1.loc[ours[0], "AP40"]; r["mAP40:95 8:1:1 (nhóm)"] = t1.loc[ours[0], "mAP40:95"]
        if paper in t1.index:
            r["AP40 8:1:1 (bài)"] = t1.loc[paper, "AP40"]; r["mAP40:95 8:1:1 (bài)"] = t1.loc[paper, "mAP40:95"]
    cmp.append(r)
table_cmp = pd.DataFrame(cmp).set_index("mô hình")
table_cmp.to_csv(RES_DIR / "table_step2_vs_811.csv")
table_cmp

## 6. Gửi kết quả

Gửi 4 file trong `results_step2/` vào thread để Claude tick mục 2.1 đến 2.3 và viết phần nhận xét:
`table_step2_main.csv`, `table_step2_per_class.csv`, `table_step2_paired.csv`, `table_step2_vs_811.csv`.